# Notebook 29: Variational Autoencoders (VAE)

---

## What This Notebook Covers

This notebook builds a **Variational Autoencoder** from the ground up, motivated by a concrete failure of the ordinary autoencoder: *a plain autoencoder can reconstruct, but it cannot generate.* We first build a standard MLP autoencoder, watch it reconstruct Fashion-MNIST beautifully and then produce **garbage** when we feed it random latents, and diagnose exactly why. The VAE fixes that failure by shaping the latent space into a known distribution we *can* sample from. We will learn:

1. **The autoencoder → generation gap** — why a network that reconstructs perfectly still can't be sampled from
2. **The VAE architecture** — an encoder that outputs a *distribution* (`mu`, `logvar`) instead of a point, and a decoder that reconstructs from a sample
3. **The reparameterization trick** — `z = mu + σ·ε` — the one idea that makes sampling differentiable so the whole thing trains by backprop
4. **The KL-divergence regularizer** — the closed-form penalty that pulls the latent distribution toward a unit Gaussian, and why that is exactly what makes the space sampleable
5. **The loss as a negative ELBO** — reconstruction + KL, read through the lens of variational inference
6. **Sampling** — draw `z ~ N(0, I)`, decode, and now get plausible images

---

## Why VAEs?

An autoencoder squeezes an image through a bottleneck and reconstructs it. After training, its **decoder** is a function from a low-dimensional latent vector to an image — which *looks* like a generator. So why not just feed it random latent vectors and get new images?

Because **nothing constrained the latent space to be nice.** The encoder is free to scatter training images anywhere in the 200-dimensional latent space — clumps here, empty voids there, arbitrary scales per axis. The decoder only ever learned to invert the *specific* points the encoder produced. Sample a random vector and you almost surely land in a void the decoder has never seen, and out comes noise. We will *see* this failure explicitly below.

The VAE's fix is elegant: **force the encoder's outputs to look like samples from a fixed, known distribution** — a standard Gaussian `N(0, I)` — by (a) having the encoder output a *distribution* per input rather than a point, and (b) adding a penalty that keeps those distributions close to `N(0, I)`. Once the aggregate latent distribution is (approximately) a unit Gaussian, sampling is trivial: draw `z ~ N(0, I)`, decode, done. The empty voids are gone because the whole space is now covered by the prior.

**For your background (this is a real, deep bridge, not a forced one).** A VAE *is* **amortized variational inference**. The encoder is a learned, amortized approximate posterior `q_φ(z|x)`; the prior is `p(z) = N(0, I)`; the decoder is the likelihood `p_θ(x|z)`; and the training loss is the **negative ELBO**, `−E_q[log p(x|z)] + KL(q(z|x) ‖ p(z))`. Everything you know from Bayesian stats — the ELBO, the reconstruction-vs-regularization tension, the KL to a prior, pathwise (reparameterization) gradient estimators vs. score-function estimators — is *exactly* what is happening here, just with neural networks standing in for the distributions. I will make that mapping explicit in the deep dives.

**Climate / EO bridges (real ones).**
- **VAE latent space ↔ nonlinear EOF / reduced-order model.** EOF/PCA gives you a *linear* low-dimensional representation of a spatial field; a VAE gives you a *nonlinear, probabilistic* one, with a latent space you can sample and interpolate. This is the basis of generative reduced-order emulators for climate fields.
- **The VAE here is the front half of latent diffusion.** Stable Diffusion and the latent-diffusion notebooks (30–31) train a diffusion model in a VAE's latent space rather than pixel space. This notebook builds exactly that compression stage.

---

## Prerequisites

You should be comfortable with:

- **Autoencoders** (notebook 08) — encoder/decoder, the reconstruction bottleneck.
- **The `miniai` training loop** (notebooks 09–12) — `Learner`, callbacks, `MetricsCB`, `OneCycleLR`, `MixedPrecision`, `lr_find`.
- **BCE loss for pixel reconstruction** — `binary_cross_entropy_with_logits` treating each pixel as an independent Bernoulli.
- **Gaussian distributions and (ideally) the ELBO / KL divergence** — helpful but the deep dives derive what we need.

---


# Part 1: Setup and Data

Imports (a large but standard stack), reproducibility settings, and the data pipeline. The one thing to notice in the data setup is a small trick: for an autoencoder the **target is the input**, so the transform sets both `b[xl]` and `b[yl]` to the same flattened image.

---


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='1'

**What does the code above do?**

Pins training to GPU index 1 before any CUDA context exists. On a single-GPU machine set this to `'0'` or remove it.


In [ ]:
import timm, torch, random, datasets, math, fastcore.all as fc, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
import k_diffusion as K, torchvision.transforms as T
import torchvision.transforms.functional as TF,torch.nn.functional as F

from torch.utils.data import DataLoader,default_collate
from pathlib import Path
from torch.nn import init
from fastcore.foundation import L
from torch import nn,tensor
from datasets import load_dataset
from operator import itemgetter
from torcheval.metrics import MulticlassAccuracy,Mean,Metric
from functools import partial
from torch.optim import lr_scheduler
from torch import optim
from einops import rearrange

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.training import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *
from miniai.augment import *
from miniai.accel import *

**What does the code above do?**

The standard course import block, pulling in PyTorch, the scientific stack, and the whole accumulated `miniai` library (datasets, learner, callbacks, init, schedules, etc.). Note `from torcheval.metrics import ..., Mean, Metric` — we subclass `Mean` later to log the two loss components separately. `k_diffusion` and `timm` are imported by habit but not central to this notebook.


In [ ]:
torch.set_printoptions(precision=4, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

import logging
logging.disable(logging.WARNING)

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

**What does the code above do?**

Reproducibility and display boilerplate: readable tensor printing, reversed-gray colormap (so Fashion-MNIST renders naturally), modest DPI, silenced warnings, a fixed seed, and an 8-worker cap for data loading.


In [ ]:
xl,yl = 'image','label'
name = "fashion_mnist"
bs = 256
dsd = load_dataset(name)

**What does the code above do?**

Loads Fashion-MNIST. `xl='image'`, `yl='label'` name the dataset keys; batch size 256. `dsd` is a `DatasetDict` with `train`/`test`.


In [ ]:
@inplace
def transformi(b):
    img = [TF.to_tensor(o).flatten() for o in b[xl]]
    b[yl] = b[xl] = img

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=8)

**What does the code above do?**

The data transform, with the autoencoder-defining trick highlighted:

- `TF.to_tensor(o).flatten()` converts each 28×28 image to a **flat 784-vector** in `[0, 1]` (this is an MLP autoencoder, so we work with vectors, not 2D maps).
- **`b[yl] = b[xl] = img`** — the crucial line: the *label* (target `y`) is set equal to the *image* (input `x`). An autoencoder's job is to reproduce its input, so the "correct answer" for each image is the image itself. This is what turns a supervised training loop into self-supervised reconstruction.

`with_transform` applies it lazily; `DataLoaders.from_dd` builds train/valid loaders.


In [ ]:
dl = dls.valid
xb,yb = b = next(iter(dl))

**What does the code above do?**

Grabs one validation batch. Because of the transform, `xb` and `yb` are identical (both the flattened images) — `xb.shape` is `(256, 784)`. We keep `xb` around to visualize reconstructions later.


---

# Part 2: A Plain Autoencoder

Before the VAE, we build an ordinary autoencoder to establish the baseline — and, more importantly, to *demonstrate the failure* the VAE exists to fix. It is a symmetric MLP: encoder `784 → 400 → 400 → 200`, decoder `200 → 400 → 400 → 784`.

---


In [ ]:
ni,nh,nl = 784,400,200

**What does the code above do?**

Names the three sizes used throughout: `ni=784` (input/output = flattened pixels), `nh=400` (hidden width), `nl=200` (latent/bottleneck dimension). The `200`-dim bottleneck is the compressed code.


In [ ]:
def lin(ni, nf, act=nn.SiLU, norm=nn.BatchNorm1d, bias=True):
    layers = nn.Sequential(nn.Linear(ni, nf, bias=bias))
    if act : layers.append(act())
    if norm: layers.append(norm(nf))
    return layers

**What does the code above do?**

A factory for a `Linear → activation → norm` block (activation and norm optional). Defaults: `SiLU` activation and `BatchNorm1d`. Unlike the *pre*-activation `lin` of notebook 28, here the order is `Linear` **then** `act`/`norm` (post-activation) — fine for this shallow MLP. Passing `act=None` gives a bare linear layer, used for the final decoder layer (which must output unbounded logits) and the VAE's `mu`/`logvar` heads.


In [ ]:
def init_weights(m, leaky=0.):
    if isinstance(m, (nn.Conv1d,nn.Conv2d,nn.Conv3d,nn.Linear)): init.kaiming_normal_(m.weight, a=leaky)

In [ ]:
iw = partial(init_weights, leaky=0.2)

**What does the code above do?**

`init_weights` applies **Kaiming-normal** initialization to every conv/linear layer's weights, with a negative-slope parameter `a=leaky` matching the activation. `iw = partial(..., leaky=0.2)` fixes that slope to 0.2 (a mild leaky-ReLU-like assumption that pairs reasonably with `SiLU`). Calling `iw(self)` on a module recursively initializes all its sublayers — good initialization keeps activations well-scaled at the start of training (notebook 11's lesson).


In [ ]:
class Autoenc(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(lin(ni, nh), lin(nh, nh), lin(nh, nl))
        self.dec = nn.Sequential(lin(nl, nh), lin(nh, nh), lin(nh, ni, act=None))
        iw(self)

    def forward(self, x):
        x = self.enc(x)
        return self.dec(x)

**What does the code above do?**

The autoencoder. The **encoder** maps `784 → 400 → 400 → 200`, compressing each image to a 200-d code. The **decoder** mirrors it back `200 → 400 → 400 → 784`, with `act=None` on the final layer so it emits raw **logits** (later passed through a sigmoid to get pixel probabilities, or fed straight to `BCEWithLogitsLoss`). `iw(self)` initializes all weights. `forward` is simply encode-then-decode; the 200-d code in the middle is the latent representation.


In [ ]:
opt_func = partial(optim.Adam, eps=1e-5)

**What does the code above do?**

Fixes the optimizer to Adam with a slightly larger `eps=1e-5` (a numerical-stability tweak common with mixed precision). Reused for both the autoencoder and the VAE.


In [ ]:
Learner(Autoenc(), dls, nn.BCEWithLogitsLoss(), cbs=[DeviceCB(), MixedPrecision()], opt_func=opt_func).lr_find()

**What does the code above do?**

Runs the learning-rate finder on a fresh autoencoder. The loss is **`BCEWithLogitsLoss`**: each of the 784 output logits is treated as an independent Bernoulli prediction of that pixel's intensity (pixels are in `[0,1]`, so BCE is a sensible reconstruction loss for grayscale). **What you should see:** the classic LR-finder curve — loss flat, then descending, then exploding — suggesting a good `max_lr` around `1e-2`–`3e-2`. (Plot omitted.)


In [ ]:
lr = 3e-2
epochs = 20
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = Autoenc()
learn = Learner(model, dls, nn.BCEWithLogitsLoss(), lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

Standard training setup: `max_lr=3e-2`, 20 epochs, one-cycle schedule, BCE reconstruction loss, the usual callbacks (device, live plot, metrics, per-batch schedule, mixed precision). Nothing autoencoder-specific except that the targets are the inputs.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains the autoencoder for 20 epochs. **What you should see:** BCE loss falling steadily as the network learns to reconstruct Fashion-MNIST. (Plot/metrics omitted.)


---

# Part 3: The Autoencoder Reconstructs — But Can't Generate

Here is the crux of the whole notebook. We check two things: (1) does the trained autoencoder reconstruct its inputs? (yes, well). (2) can we *generate* new images by decoding random latents? (no — garbage). The gap between these two is precisely what the VAE closes.

---


In [ ]:
with torch.no_grad(): t = to_cpu(model(xb.cuda()).float())

**What does the code above do?**

Runs the validation batch through the trained autoencoder (no gradients, moved to CPU). `t` holds the reconstructed logits, ready to compare against the originals.


In [ ]:
show_images(xb[:9].reshape(-1,1,28,28), imsize=1.5, title='Original');

**What does the code above do?**

Shows the first 9 **original** images (reshaped from flat 784-vectors back to 28×28). **What you should see:** nine clear Fashion-MNIST items. (Image omitted.)


In [ ]:
show_images(t[:9].reshape(-1,1,28,28).sigmoid(), imsize=1.5, title='Autoenc');

**What does the code above do?**

Shows the autoencoder's **reconstructions** of those same 9 images (`.sigmoid()` turns logits into `[0,1]` pixels). **What you should see:** nine reconstructions that closely match the originals — perhaps slightly blurred, but clearly the same items. The autoencoder reconstructs well. So far so good.


In [ ]:
noise = torch.randn(16, nl).cuda()
with torch.no_grad(): generated_images = model.dec(noise).sigmoid()

**What does the code above do?**

The critical experiment: **skip the encoder** and feed the decoder 16 random latent vectors `z ~ N(0, I)` directly, as if the decoder were a generator. If the latent space were a nice unit Gaussian, this would produce plausible images.


In [ ]:
show_images(generated_images.reshape(-1, 1, 28, 28), imsize=1.5)

**What does the code above do?**

Shows those 16 decoded-from-noise images. **What you should see:** **garbage** — blurry, structureless blobs, nothing like Fashion-MNIST. This is the failure that motivates the VAE.

**Why it fails (the key insight).** The plain autoencoder was never told *where* to put its codes. During training the encoder scattered the 200-d codes into some unknown, irregular cloud — clusters, gaps, wildly different scales per axis. The decoder only learned to invert points *in that cloud*. A standard-normal sample `randn(16, 200)` almost certainly lands **outside** the cloud, in a region the decoder has never seen, so it outputs mush. Reconstruction (encode a real image, decode it) stays inside the cloud and works; generation (decode a random point) leaves the cloud and fails.

The fix is not a bigger decoder — it is to **make the code cloud coincide with the distribution we sample from.** That is the VAE.


---

# Part 4: The Variational Autoencoder

The VAE changes two things. First, the encoder no longer outputs a single point; it outputs the **parameters of a Gaussian** — a mean `mu` and a log-variance `lv` — one per latent dimension. We then **sample** the code `z` from that Gaussian. Second, we add a **KL-divergence penalty** that pulls each per-input Gaussian toward the standard normal `N(0, I)`. Together these force the aggregate latent distribution to be (approximately) `N(0, I)`, so decoding a random `N(0, I)` sample now works.

---


In [ ]:
# sd vae is 3 down, 1 no-down, mid, conv, sampling, conv, mid, 3 up, 1 no-up

**What does the code above do?**

A comment noting the architecture of the **Stable Diffusion VAE** for context: a convolutional encoder (3 downsampling stages + a non-downsampling stage + a mid block), then the sampling bottleneck, then a mirror decoder. Our VAE here is a tiny MLP version of the same idea — the *mechanism* (encode to a distribution, sample, decode, regularize with KL) is identical; only the layers differ. This is the compression model latent diffusion is built on.


In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(lin(ni, nh), lin(nh, nh))
        self.mu,self.lv = lin(nh, nl, act=None),lin(nh, nl, act=None)
        self.dec = nn.Sequential(lin(nl, nh), lin(nh, nh), lin(nh, ni, act=None))
        iw(self)

    def forward(self, x):
        x = self.enc(x)
        mu,lv = self.mu(x),self.lv(x)
        z = mu + (0.5*lv).exp()*torch.randn_like(lv)
        return self.dec(z),mu,lv

**What does the code above do?**

The VAE. Compared to `Autoenc`, the encoder body stops one layer early (`784 → 400 → 400`) and then splits into **two heads**:

- `self.mu = lin(nh, nl, act=None)` — outputs the latent **mean** vector (200-d).
- `self.lv = lin(nh, nl, act=None)` — outputs the latent **log-variance** vector (200-d).

So the encoder maps each image to a *distribution* over codes, `N(mu, σ²)` with `σ² = exp(lv)`, one Gaussian per latent dimension. The forward pass then:

1. computes `mu` and `lv`,
2. **samples** a code via the reparameterization trick `z = mu + (0.5*lv).exp() * randn_like(lv)` (deep dive below — this is `z = mu + σ·ε`),
3. decodes `z`,
4. returns `(x_hat, mu, lv)` — the reconstruction *and* the distribution parameters, because the loss needs all three.

The decoder is identical to the autoencoder's. The only architectural change is "one head → two heads + a sampling step."


In [ ]:
def kld_loss(inp, x):
    x_hat,mu,lv = inp
    return -0.5 * (1 + lv - mu.pow(2) - lv.exp()).mean()

def bce_loss(inp, x): return F.binary_cross_entropy_with_logits(inp[0], x)

def vae_loss(inp, x): return kld_loss(inp, x) + bce_loss(inp,x)

**What does the code above do?**

The two-part VAE loss:

- **`bce_loss`** — the **reconstruction** term: BCE between the decoded logits `inp[0]` and the target image `x`. "Did we rebuild the input?"
- **`kld_loss`** — the **regularization** term: `−0.5·mean(1 + lv − mu² − exp(lv))`, the closed-form KL divergence from each per-input Gaussian `N(mu, σ²)` to the prior `N(0, I)`. "Is the latent distribution close to a unit Gaussian?" (Full derivation in the deep dive.)
- **`vae_loss`** — their sum. This sum is the **negative ELBO**: minimizing it simultaneously maximizes reconstruction likelihood and keeps the approximate posterior near the prior.

The tension between the two terms is the entire story: BCE alone would let the encoder spread codes anywhere (the autoencoder failure); KL alone would collapse everything to `N(0, I)` and ignore the image. Balanced, they produce a latent space that is both *informative* (reconstructs) and *sampleable* (matches the prior).


## Deep Dive: The Reparameterization Trick

The VAE's forward pass contains a step that looks innocuous but is the single idea that makes VAEs trainable:

```python
z = mu + (0.5*lv).exp() * torch.randn_like(lv)
```

### The problem: you can't backprop through a random sample

We want `z` to be a *sample* from `N(mu, σ²)`, where `mu` and `σ` are produced by the encoder. The naive way to write that is `z ~ N(mu, σ²)` — call a sampler. But training needs `∂loss/∂mu` and `∂loss/∂σ` to flow back into the encoder, and **you cannot differentiate through a sampling operation**: `z = sample(N(mu, σ²))` has no useful derivative w.r.t. `mu` and `σ` because the randomness is baked into the opaque sampler. Gradient blocked, encoder untrainable.

### The fix: move the randomness outside the gradient path

Reparameterize the sample as a **deterministic, differentiable function** of `mu`, `σ`, and an *external* noise source `ε`:

$$z = \mu + \sigma \odot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I).$$

This is *the same distribution* — a Gaussian with mean `mu` and standard deviation `σ` — but now the only random thing, `ε`, is an **input**, not something we differentiate through. `mu` and `σ` enter via plain arithmetic (`+` and `×`), which backprop handles trivially:

$$\frac{\partial z}{\partial \mu} = 1, \qquad \frac{\partial z}{\partial \sigma} = \varepsilon.$$

Gradients now flow cleanly from the loss, through `z`, into both encoder heads. (In the language of your Bayesian background: this is the **pathwise / reparameterization gradient estimator**, the low-variance alternative to the score-function/REINFORCE estimator — and it is *why* VAEs train stably where a naive stochastic node would not.)

### Reading the code

```python
z = mu + (0.5*lv).exp() * torch.randn_like(lv)
```

- `torch.randn_like(lv)` is `ε ~ N(0, I)`, drawn fresh each forward pass, same shape as the latent.
- `(0.5*lv).exp()` is `σ`. Why? The encoder outputs **log-variance** `lv = log(σ²)`, so `σ = exp(½·log σ²) = exp(0.5·lv)`. (Why log-variance? See below.)
- `mu + σ·ε` is the reparameterized sample.

So this one line is `z = μ + σ·ε` — a differentiable Gaussian sample.

### Why parameterize log-variance instead of variance or σ?

Three reasons, all practical:

1. **Unconstrained output.** Variance and σ must be **positive**, but a `Linear` layer outputs any real number. `lv = log σ²` lives on all of ℝ, so the `self.lv` head can output *anything* and `exp` maps it to a valid positive σ — no clamping, no softplus, no risk of a negative variance.
2. **Numerical stability.** `exp` of a moderate log-variance stays well-behaved; and the KL term (next deep dive) is *linear* in `lv`, which keeps its gradients tame across many orders of magnitude of variance.
3. **It matches the KL's natural form.** As we will see, the closed-form KL is cleanest expressed in terms of `lv` directly.

**One subtlety worth flagging honestly:** `(0.5*lv).exp()` computes `σ = exp(½ lv) = exp(½ log σ²) = √(σ²) = σ`. Some readers expect to see a `√` somewhere for "standard deviation"; the `√` is *implicit* in the `0.5` factor inside the exp. That is correct, not a bug.


## Deep Dive: The KL Divergence Loss (and the ELBO)

The reparameterization trick lets us sample differentiably, but nothing yet forces the latent distribution to be `N(0, I)` — without that, we are back to the autoencoder's unsampleable cloud. The **KL term** supplies the force.

### What we are minimizing

For each input, the encoder gives an approximate posterior `q(z|x) = N(mu, σ²)` (diagonal covariance — each latent dimension independent). We want it close to the prior `p(z) = N(0, I)`. "Close" is measured by the **Kullback–Leibler divergence** `KL(q ‖ p)`. For two diagonal Gaussians this has a famous closed form. Per latent dimension `j`:

$$\mathrm{KL}\big(\mathcal{N}(\mu_j, \sigma_j^2)\,\|\,\mathcal{N}(0,1)\big) = \tfrac{1}{2}\left(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\right).$$

Summing over dimensions and substituting `lv_j = log σ_j²` (so `σ_j² = exp(lv_j)`):

$$\mathrm{KL} = \tfrac{1}{2}\sum_j \big(\mu_j^2 + e^{lv_j} - lv_j - 1\big) = -\tfrac{1}{2}\sum_j \big(1 + lv_j - \mu_j^2 - e^{lv_j}\big).$$

Which is **exactly** the code:

```python
-0.5 * (1 + lv - mu.pow(2) - lv.exp()).mean()
```

(`.mean()` instead of `.sum()` just rescales the term relative to BCE; the shape of the objective is unchanged.)

### Term-by-term intuition

Group the KL as a **mean penalty** and a **variance penalty**:

- **`mu²`** — penalizes the mean for drifting from 0. Pulls every input's latent code toward the origin. Without it, the encoder could encode class identity purely in far-apart means and ignore the prior.
- **`exp(lv) − lv − 1`** — penalizes the variance for departing from 1. This is minimized (equals 0) exactly at `lv = 0`, i.e. `σ² = 1`. If the encoder tries to make variance tiny (`lv → −∞`, a near-deterministic code — sneaking back toward a plain autoencoder), `−lv` blows up. If it makes variance huge (`lv → +∞`), `exp(lv)` blows up. So unit variance is the unique happy point.

Together: **push means toward 0, push variances toward 1** — i.e. shape `q(z|x)` toward `N(0, I)`. Do that for every input and the *aggregate* latent distribution becomes ≈`N(0, I)`, which is what makes `randn(n, nl)` a valid thing to decode.

### The whole loss is the negative ELBO

Variational inference maximizes the **Evidence Lower BOund**:

$$\mathrm{ELBO} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{\mathrm{KL}(q(z|x)\,\|\,p(z))}_{\text{regularizer}}.$$

Minimizing `vae_loss = bce_loss + kld_loss` is minimizing `−ELBO`:

- `bce_loss` = `−E_q[log p(x|z)]` — the negative reconstruction log-likelihood (Bernoulli per pixel ⇒ BCE), estimated with the single reparameterized sample `z`.
- `kld_loss` = `KL(q ‖ p)` — the regularizer, in closed form (no sampling needed).

So a VAE is *amortized variational inference*: the encoder amortizes the per-datapoint variational optimization into one forward pass, and we optimize the ELBO by SGD. Everything you know about the reconstruction-vs-KL trade-off, posterior collapse (KL term dominating ⇒ the model ignores `z`), and the role of the prior transfers directly.


---

# Part 5: Visualizing the Variance Penalty

The next cell plots the variance part of the KL as a function of log-variance, making the "unit variance is the sweet spot" argument visual.

---


In [ ]:
x = torch.linspace(-3,3,100)
plt.figure(figsize=(4,3))
plt.plot(x, -0.5*(1+x-x.exp()));

**What does the code above do?**

Plots `−0.5·(1 + lv − exp(lv))` — the KL's **variance term** as a function of log-variance `lv` (holding `mu=0`). **What you should see:** a convex curve with its **minimum of 0 at `lv = 0`** (i.e. variance = 1), rising on both sides — sharply for `lv > 0` (variance too big, `exp` dominates) and steadily for `lv < 0` (variance too small, `−lv` dominates).

This is the KL's variance penalty made visible: it is a bowl centered on unit variance. The encoder pays nothing at `σ²=1` and pays increasingly to move away in either direction — which is exactly the pressure that keeps the latent space a well-behaved unit Gaussian rather than collapsing to near-zero variance (a deterministic autoencoder) or blowing up.


**Question**: What would happen if the variance of the latents were very low? What if they were very high?

**Bing**: If the variance of the latents were very low, then the encoder distribution would be very peaked and concentrated around the mean. This would make the latent space less diverse and expressive, and limit the ability of the decoder to reconstruct the data accurately. It would also make it harder to generate new data that are different from the training data.

If the variance of the latents were very high, then the encoder distribution would be very spread out and diffuse. This would make the latent space more noisy and random, and reduce the correlation between the latent codes and the data. It would also make it easier to generate new data that are unrealistic or nonsensical.

---

**A sharper answer (tying it to the mechanics above).** Jeremy left the original "Bing" answer in the notebook; here is a more precise version grounded in what the loss actually does:

- **Variance → 0 (`lv → −∞`).** The reparameterized sample `z = mu + σ·ε` collapses to `z ≈ mu` — the model becomes a *deterministic* autoencoder again. Reconstruction may look great, but you lose the smooth, sampleable latent space (adjacent codes no longer decode to similar images), and the KL's `−lv` term → +∞ *punishes* this, so the loss actively resists it. This near-collapse is the failure mode the VAE is designed to avoid.
- **Variance → ∞ (`lv → +∞`).** The sampled `z` is dominated by noise `σ·ε`, so the decoder receives an almost input-independent random vector — reconstruction degrades toward the data mean (blurry, generic images), and the KL's `exp(lv)` term → +∞ punishes *this*. In VI terms, `q(z|x)` stops depending on `x` — a form of **posterior collapse**.
- **The balance.** Because both extremes are penalized and the reconstruction term needs *some* usable signal in `z`, training settles at variances near 1 with means that encode just enough about each image to reconstruct it — an informative *and* sampleable latent space.


---

# Part 6: Training the VAE

Same training machinery as the autoencoder, with two additions: a custom metric so we can watch the KLD and BCE terms separately, and the composite `vae_loss`.

---


In [ ]:
class FuncMetric(Mean):
    def __init__(self, fn, device=None):
        super().__init__(device=device)
        self.fn = fn

    def update(self, inp, targets):
        self.weighted_sum += self.fn(inp, targets)
        self.weights += 1

**What does the code above do?**

A small `torcheval` metric that averages *any* function of `(inp, targets)` over an epoch. Subclassing `Mean`, it accumulates `fn(inp, targets)` into `weighted_sum` and counts batches in `weights`, so the running mean is `weighted_sum / weights`. We use it to log `kld_loss` and `bce_loss` **separately** during training — essential for a VAE, because a single combined loss hides whether the model is trading reconstruction for regularization (or collapsing).


In [ ]:
metrics = MetricsCB(kld=FuncMetric(kld_loss), bce=FuncMetric(bce_loss))
opt_func = partial(optim.Adam, eps=1e-5)

**What does the code above do?**

Builds a `MetricsCB` that reports two named metrics each epoch: `kld` (the KL term) and `bce` (the reconstruction term), each wrapped in a `FuncMetric`. Now the training table shows both components alongside the total loss, so we can see the balance between them evolve.


In [ ]:
lr = 3e-2
epochs = 20
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), metrics, BatchSchedCB(sched), MixedPrecision()]
model = VAE()
learn = Learner(model, dls, vae_loss, lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

Assembles the VAE training run: the `VAE` model, the composite `vae_loss`, the two-component `metrics` callback, and the same one-cycle/mixed-precision setup as before. Everything the `Learner` needs is unchanged from the autoencoder except the model and loss — the reparameterization sampling happens *inside* `VAE.forward`, invisible to the training loop.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains the VAE for 20 epochs. **What you should see:** the total loss falling, with the `bce` (reconstruction) and `kld` (regularization) columns both visible — typically BCE dominates in magnitude while KLD settles to a small, stable value as the latent distribution locks onto ≈`N(0, I)`. (Plot/metrics omitted.)


---

# Part 7: The VAE Reconstructs *and* Generates

Now we repeat Part 3's two experiments on the VAE. Reconstruction should still be good, and — the payoff — decoding random `N(0, I)` latents should now produce plausible Fashion-MNIST, because the KL term made the latent space match the sampling distribution.

---


In [ ]:
with torch.no_grad(): t,mu,lv = to_cpu(model(xb.cuda()))
t = t.float()

**What does the code above do?**

Runs the validation batch through the VAE. Note `forward` returns a **triple** `(x_hat, mu, lv)`, so we unpack all three; `t = x_hat` is the reconstruction (cast to float from mixed-precision half). `mu`/`lv` are available if we want to inspect the latent statistics.


In [ ]:
show_images(xb[:9].reshape(-1,1,28,28), imsize=1.5, title='Original');

**What does the code above do?**

The 9 original images again, for side-by-side comparison. (Image omitted.)


In [ ]:
show_images(t[:9].reshape(-1,1,28,28).sigmoid(), imsize=1.5, title='VAE');

**What does the code above do?**

The VAE's reconstructions. **What you should see:** recognizable reconstructions of the originals — typically a touch **blurrier** than the plain autoencoder's, which is the characteristic VAE trade-off: the KL regularization and the sampling noise cost some reconstruction sharpness in exchange for a well-structured latent space. (Image omitted.)


In [ ]:
noise = torch.randn(16, nl).cuda()
with torch.no_grad(): ims = model.dec(noise).sigmoid()

**What does the code above do?**

The same generation experiment that failed for the autoencoder: decode 16 random `N(0, I)` latents directly. This time the latent space *is* approximately `N(0, I)` (thanks to the KL term), so these random points fall where the decoder has learned to operate.


In [ ]:
show_images(ims.reshape(-1, 1, 28, 28), imsize=1.5)

**What does the code above do?**

Shows the 16 images generated from pure noise. **What you should see:** **plausible Fashion-MNIST items** — bags, shirts, shoes, trousers — not the garbage the autoencoder produced. They may be soft or slightly generic, but they are recognizably *clothes*.

This is the whole point, demonstrated: the VAE turned an un-sampleable autoencoder into a **generative model**. The only structural additions were the two encoder heads (`mu`, `lv`), the reparameterization sample, and the KL penalty — and those together reshaped the latent space into something you can draw from.


---

# Summary and What's Next

### What we built

| Step | What | Why |
|------|------|-----|
| Autoencoder | MLP enc/dec, BCE loss | Baseline: reconstructs well |
| Failure demo | `dec(randn)` → garbage | Latent space isn't sampleable |
| VAE encoder | two heads `mu`, `lv` | Output a *distribution*, not a point |
| Reparameterization | `z = mu + exp(0.5·lv)·ε` | Differentiable sampling → trainable |
| KL loss | `−0.5·(1+lv−mu²−exp(lv))` | Pull `q(z|x)` toward `N(0,I)` |
| `vae_loss` | BCE + KL = −ELBO | Reconstruct *and* regularize |
| Generation | `dec(randn)` → clothes | Latent space now matches the prior |

### The ideas to remember

1. **A good reconstructor is not a generator.** Without a constraint on the latent distribution, the decoder only knows the encoder's arbitrary cloud; random samples miss it. This is *the* motivation for the VAE.
2. **Reparameterization = move the randomness to an input.** `z = mu + σ·ε` with `ε~N(0,I)` external makes the sample a differentiable function of the encoder outputs. Pathwise gradients; the encoder trains.
3. **Log-variance for an unconstrained, stable parameterization.** `σ = exp(0.5·lv)` guarantees positivity from an unbounded linear head, and the KL is linear in `lv`.
4. **The KL is a bowl at unit variance and origin mean.** `mu²` pulls means to 0; `exp(lv)−lv−1` pulls variance to 1. Together they make the aggregate latent ≈`N(0,I)`.
5. **`vae_loss` is the negative ELBO.** Reconstruction likelihood + KL-to-prior — amortized variational inference by SGD.

### Where this goes

This VAE is the **compression front-end of latent diffusion** (notebooks 30–31). Instead of running an expensive diffusion U-Net on 32×32 (or 512×512) pixels, latent diffusion trains the VAE once to map images to a small latent, then runs the *entire* attention-conditioned diffusion process of notebook 28 in that latent space, and finally decodes with the VAE. The payoff is a large speed/memory win with little quality loss — and it is why Stable Diffusion is tractable. So notebooks 28 (the conditioned diffusion U-Net) and 29 (the VAE) are the two halves that combine into 30–31.

### Suggested next steps

1. Re-read the two deep dives (reparameterization, KL/ELBO) — and, given your Bayesian background, check my ELBO/KL framing against your own derivation; push back if the amortized-VI mapping is off anywhere.
2. Try the experiments the notebook implies: scale the KL term up/down (a "β-VAE") and watch reconstruction sharpness trade against sample quality; or interpolate between two encoded images in latent space and decode the path.
3. When satisfied, run `concept-extraction` (candidates: autoencoder-can't-generate, reparameterization trick, log-variance parameterization, closed-form Gaussian KL, VAE loss = −ELBO, KL variance-penalty bowl).
4. Optionally `/colab` for a GPU-ready version and `/html` to publish.

---
